# Notebook 2 — NumPy & Pandas : niveau INTERMÉDIAIRE

**Objectif** : combiner plusieurs notions dans des problèmes proches de situations réelles, sur des jeux de données plus volumineux.

**Pré-requis** : avoir terminé `01_DEBUTANT.ipynb` (ou en maîtriser le contenu).

**Comment travailler**
- Chaque exercice demande de **choisir** les bons outils : les consignes décrivent le résultat attendu, pas la méthode.
- Vérifiez systématiquement `shape`, `dtype`, la taille avant/après les jointures et les filtres.
- Avant de coder, décrivez en une phrase la **démarche**. Après, vérifiez avec un petit contrôle (`assert`, somme de contrôle).
- Corrections uniquement en fin de notebook.

**Contenu** : 22 exercices — NumPy avancé (broadcasting, multi-dimension, algèbre), puis Pandas (jointures, GroupBy avancé, pivots, nettoyage, dates, séries temporelles, fonctions personnalisées, optimisation) et exercices mixtes NumPy + Pandas.

**Niveaux** : 🔵 intermédiaire · 🟠 exigeant · 🔴 synthèse.

> Exécutez d'abord les cellules de préparation : elles génèrent les jeux de données (reproductibles grâce aux graines).

## Sommaire des exercices

- Exercice 1 — Matrice de distances par broadcasting *(🔵 intermédiaire)*
- Exercice 2 — Softmax par ligne et normalisation *(🔵 intermédiaire)*
- Exercice 3 — Tableau 3D : trafic web *(🟠 exigeant)*
- Exercice 4 — Portefeuille financier : algèbre matricielle *(🟠 exigeant)*
- Exercice 5 — Moyenne mobile et cumul sans boucle *(🔵 intermédiaire)*
- Exercice 6 — Régression linéaire à la main *(🟠 exigeant)*
- Exercice 7 — Affectation au centroïde le plus proche (vectorisation) *(🟠 exigeant)*
- Exercice 8 — Explorer les trois tables *(🔵 intermédiaire)*
- Exercice 9 — Jointures : construire la table d'analyse *(🟠 exigeant)*
- Exercice 10 — Chiffre d'affaires mensuel *(🔵 intermédiaire)*
- Exercice 11 — GroupBy avancé : agrégations nommées et top N par groupe *(🟠 exigeant)*
- Exercice 12 — transform et filter *(🟠 exigeant)*
- Exercice 13 — Tableau croisé dynamique *(🔵 intermédiaire)*
- Exercice 14 — Annulations et retours : analyser une proportion *(🟠 exigeant)*
- Exercice 15 — Nettoyer le fichier de ventes sale *(🟠 exigeant)*
- Exercice 16 — Segmentation RFM des clients *(🔴 synthèse)*
- Exercice 17 — Fonction personnalisée de profilage *(🔵 intermédiaire)*
- Exercice 18 — Motifs hebdomadaires et moyenne mobile *(🔵 intermédiaire)*
- Exercice 19 — Capteur : trous, anomalies et profil journalier *(🟠 exigeant)*
- Exercice 20 — Réduire la mémoire et accélérer *(🟠 exigeant)*
- Exercice 21 — Panier moyen : intervalle de confiance par bootstrap *(🔴 synthèse)*
- Exercice 22 — Analyse exploratoire complète : que raconte Kora en 2024 ? *(🔴 synthèse)*
- **CORRECTIONS ET EXPLICATIONS** (en fin de notebook — ne regardez qu'après avoir cherché !)

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## Jeux de données

### 1. Boutique en ligne « Kora » (e-commerce) — année 2024
Trois tables liées : `clients`, `catalogue`, `commandes` (chaque ligne de `commandes` = un produit commandé par un client). La table `commandes` contient volontairement quelques **clients inconnus** (références orphelines), comme dans un vrai système mal synchronisé.

In [ ]:
rng = np.random.default_rng(2024)

# ---------- Catalogue (40 produits) ----------
noms = {
    "Mode": ["Pagne wax 6 yards", "Robe cintrée", "Chemise brodée", "Sandales cuir", "Sac à main", "Foulard soie", "Casquette", "Ceinture cuir"],
    "Beauté": ["Beurre de karité", "Savon noir", "Huile de coco", "Parfum", "Vernis à ongles", "Brosse à cheveux", "Crème visage", "Gloss"],
    "Maison": ["Natte tressée", "Coussin wax", "Lampe artisanale", "Set de vaisselle", "Théière", "Panier tressé", "Rideau", "Bougie parfumée"],
    "Électronique": ["Écouteurs", "Chargeur rapide", "Powerbank", "Enceinte Bluetooth", "Montre connectée", "Câble USB", "Coque téléphone", "Clé USB 64Go"],
    "Alimentation": ["Miel local", "Mélange d'épices", "Café moulu", "Thé au citron", "Huile de palme", "Piment moulu", "Noix de cajou", "Jus de bissap"],
}
bornes = {"Mode": (3000, 25000), "Beauté": (1500, 12000), "Maison": (2000, 15000),
          "Électronique": (2500, 30000), "Alimentation": (800, 6000)}
lignes = []
for cat, liste in noms.items():
    for nom in liste:
        prix = int(round(rng.uniform(*bornes[cat]), -2))
        lignes.append((nom, cat, prix, int(round(prix * rng.uniform(0.55, 0.75), -1))))
catalogue = pd.DataFrame(lignes, columns=["nom", "categorie", "prix_unitaire", "cout"])
catalogue.insert(0, "id_produit", np.arange(1, len(catalogue) + 1))

# ---------- Clients (1 500) ----------
villes = ["Cotonou", "Abomey-Calavi", "Porto-Novo", "Parakou", "Bohicon", "Ouidah"]
n_cli = 1500
clients = pd.DataFrame({
    "id_client": np.arange(1, n_cli + 1),
    "ville": rng.choice(villes, n_cli, p=[0.35, 0.22, 0.15, 0.12, 0.09, 0.07]),
    "age": np.clip(rng.normal(31, 8, n_cli).round(), 18, 65).astype(int),
    "canal": rng.choice(["Instagram", "Facebook", "WhatsApp", "Google", "Bouche-à-oreille"], n_cli, p=[0.30, 0.25, 0.20, 0.10, 0.15]),
    "date_inscription": pd.Timestamp("2023-06-01") + pd.to_timedelta(rng.integers(0, 518, n_cli), unit="D"),
})

# ---------- Commandes : chaque client passe un nombre variable de commandes, après son inscription ----------
activite = rng.gamma(2.0, 1.0, n_cli)
nb_cmd = 1 + rng.poisson(activite * 1.6)
id_cli = np.repeat(clients["id_client"].to_numpy(), nb_cmd)
debut = np.maximum(np.repeat(clients["date_inscription"].to_numpy(), nb_cmd), np.datetime64("2024-01-01"))
jours = pd.date_range("2024-01-01", "2024-12-31")
poids_jour = 1 + 0.7 * jours.month.isin([11, 12]) + 0.25 * (jours.dayofweek >= 5) + 0.3 * (jours.month == 6)
cdf = np.cumsum(poids_jour) / poids_jour.sum()
idx_debut = (debut - np.datetime64("2024-01-01")).astype("timedelta64[D]").astype(int)
lo = np.where(idx_debut > 0, cdf[np.clip(idx_debut - 1, 0, None)], 0.0)
u = lo + rng.random(len(id_cli)) * (1 - lo)
date_cmd = jours[np.clip(np.searchsorted(cdf, u), 0, len(jours) - 1)]
keep = idx_debut < len(jours)                       # clients inscrits après le 31/12/2024 : aucune commande
id_cli, date_cmd = id_cli[keep], date_cmd[keep]
n = len(id_cli)

popularite = rng.dirichlet(np.ones(len(catalogue)) * 1.5)
paiement = rng.choice(["Mobile Money", "Cash à la livraison", "Carte", "Virement"], n, p=[0.55, 0.30, 0.10, 0.05])
p_annul = np.where(paiement == "Cash à la livraison", 0.13, 0.05)
tirage = rng.random(n)
statut = np.where(tirage < p_annul, "Annulée", np.where(tirage < p_annul + 0.06, "Retournée", "Livrée"))
commandes = pd.DataFrame({
    "id_commande": np.arange(50001, 50001 + n),
    "id_client": id_cli,
    "id_produit": rng.choice(catalogue["id_produit"], n, p=popularite),
    "date": date_cmd,
    "quantite": rng.choice([1, 2, 3, 4], n, p=[0.70, 0.20, 0.07, 0.03]),
    "mode_paiement": paiement,
    "statut": statut,
})
orphelines = commandes.sample(15, random_state=3).copy()
orphelines["id_client"] = rng.integers(9000, 9100, len(orphelines))
orphelines["id_commande"] = np.arange(90001, 90001 + len(orphelines))
commandes = pd.concat([commandes, orphelines]).sort_values("date").reset_index(drop=True)
print(clients.shape, catalogue.shape, commandes.shape)

### 2. Fichier de ventes « sale »
Export manuel d'un tableur : noms de colonnes irréguliers, dates en deux formats, montants en texte (`"12 500 FCFA"`), quantités manquantes ou négatives, valeurs aberrantes, doublons, villes aux graphies multiples.

In [ ]:
rng_b = np.random.default_rng(7)
m = 600
base = pd.DataFrame({
    "ref": [f"CMD-{i:04d}" for i in range(1, m + 1)],
    "date": pd.Timestamp("2024-03-01") + pd.to_timedelta(rng_b.integers(0, 120, m), unit="D"),
    "ville": rng_b.choice(villes, m),
    "produit": rng_b.choice(catalogue["nom"], m),
    "quantite": rng_b.choice([1, 2, 3, 4, 5], m, p=[0.5, 0.25, 0.12, 0.08, 0.05]),
}).merge(catalogue[["nom", "prix_unitaire"]], left_on="produit", right_on="nom").drop(columns="nom")
base = base.sort_values("ref").reset_index(drop=True)

def variante(v):
    return rng_b.choice([v, v.upper(), v.lower(), " " + v + " "])

bruts = pd.DataFrame({
    "Réf": base["ref"],
    "Date vente": np.where(rng_b.random(m) < 0.85, base["date"].dt.strftime("%Y-%m-%d"), base["date"].dt.strftime("%d/%m/%Y")),
    "Ville ": [variante(v) for v in base["ville"]],
    "Produit": base["produit"],
    "Quantité": base["quantite"].astype(float),
    "Montant": (base["quantite"] * base["prix_unitaire"]).map(lambda v: f"{v:,}".replace(",", " ") + " FCFA"),
})
idx = rng_b.choice(m, 37, replace=False)
bruts.loc[idx[:15], "Quantité"] = np.nan
bruts.loc[idx[15:21], "Quantité"] = -1
bruts.loc[idx[21:33], "Montant"] = "N/A"
bruts.loc[idx[33:], "Montant"] = "9 999 999 FCFA"
bruts = pd.concat([bruts, bruts.sample(20, random_state=1)], ignore_index=True)
print(bruts.shape)
bruts.head()

### 3. Capteur de consommation électrique (série temporelle horaire, 90 jours)
Avec un profil journalier réaliste (pics du matin et du soir), un effet week-end, des **pics anormaux** et des **trous** de mesure.

In [ ]:
rng_c = np.random.default_rng(11)
idx_h = pd.date_range("2024-09-01", periods=24 * 90, freq="h")
h = idx_h.hour.to_numpy(); dow = idx_h.dayofweek.to_numpy()
profil = 2.0 + 1.5 * np.exp(-((h - 19) ** 2) / 8) + 0.8 * np.exp(-((h - 7) ** 2) / 6)
profil = profil * np.where(dow >= 5, 1.15, 1.0)
conso = profil + rng_c.normal(0, 0.15, len(idx_h))
conso[rng_c.choice(len(idx_h), 6, replace=False)] += rng_c.uniform(4, 7, 6)
temp = 27 + 4 * np.sin((h - 9) / 24 * 2 * np.pi) + rng_c.normal(0, 0.8, len(idx_h))
capteur = pd.DataFrame({"conso_kwh": conso.round(2), "temp_ext": temp.round(1)}, index=idx_h)
capteur.iloc[500:530, 0] = np.nan
capteur.iloc[1500:1503, 0] = np.nan
capteur.index.name = "horodatage"
print(capteur.shape, capteur["conso_kwh"].isna().sum())

### 4. Rendements journaliers de 5 actifs financiers (250 jours de bourse)
Simulés avec des corrélations réalistes (cacao et coton corrélés, or et pétrole faiblement liés).

In [ ]:
rng_r = np.random.default_rng(5)
actifs = ["BRVM10", "Or", "Pétrole", "Cacao", "Coton"]
mu = np.array([0.0004, 0.0003, 0.0002, 0.0005, 0.0001])
vol = np.array([0.008, 0.010, 0.018, 0.014, 0.012])
corr = np.array([[1.0, 0.2, 0.3, 0.1, 0.1],
                 [0.2, 1.0, -0.1, 0.0, 0.05],
                 [0.3, -0.1, 1.0, 0.1, 0.2],
                 [0.1, 0.0, 0.1, 1.0, 0.4],
                 [0.1, 0.05, 0.2, 0.4, 1.0]])
L = np.linalg.cholesky(corr * np.outer(vol, vol))
rendements = mu + rng_r.standard_normal((250, 5)) @ L.T          # (250 jours, 5 actifs)

villes_coord = pd.DataFrame({"ville": villes, "lat": [6.37, 6.45, 6.50, 9.34, 7.18, 6.37],
                             "lon": [2.39, 2.35, 2.63, 2.63, 2.07, 2.09]})
print(rendements.shape)

---
# PARTIE I — NumPy avancé

### Exercice 1 — Matrice de distances par broadcasting
*Niveau : 🔵 intermédiaire*

Avec `villes_coord` (latitude/longitude de 6 villes du Bénin) :

1. Construisez le tableau `P` de forme `(6, 2)` des coordonnées. On approxime les distances en kilomètres par `111 × distance euclidienne en degrés`.
2. Calculez, **sans aucune boucle**, la matrice `D` `(6, 6)` des distances entre toutes les paires de villes (indice : insérez des axes pour produire un tableau intermédiaire `(6, 6, 2)`).
3. Vérifiez : `D` est symétrique, de diagonale nulle.
4. Pour chaque ville, trouvez la ville **la plus proche** (autre qu'elle-même) : affichez les paires « ville → plus proche voisine (distance en km) ».
5. Quelle est la distance maximale du jeu ? Entre quelles villes ?

### Exercice 2 — Softmax par ligne et normalisation
*Niveau : 🔵 intermédiaire*

Un modèle de classification produit, pour 5 clients, des « scores » pour 4 offres commerciales :

```python
scores = np.array([[2.0, 1.0, 0.1, -1.0],
                   [0.5, 2.5, 0.3,  0.0],
                   [1.0, 1.0, 1.0,  1.0],
                   [-2., 0.0, 3.0,  1.0],
                   [900., 905., 899., 902.]])
```
1. Calculez les probabilités `softmax` de **chaque ligne** : `exp(s_ij) / Σ_j exp(s_ij)`. Vérifiez que chaque ligne somme à 1.
2. Votre implémentation naïve donne-t-elle un résultat correct pour la dernière ligne ? Pourquoi (overflow de `exp`) ?
3. Corrigez en soustrayant le maximum de chaque ligne avant l'exponentielle (astuce de stabilité). Montrez que le résultat est identique aux autres lignes.
4. Pour chaque client, quelle offre a la plus forte probabilité, et quelle est cette probabilité ?

### Exercice 3 — Tableau 3D : trafic web
*Niveau : 🟠 exigeant*

On dispose du nombre de visites d'un site, par heure, pour 4 semaines complètes :

```python
rng_t = np.random.default_rng(3)
heures = np.arange(24)
profil = 40 + 120 * np.exp(-((heures - 20) ** 2) / 10) + 60 * np.exp(-((heures - 12) ** 2) / 12)
jour_facteur = np.array([1.0, 1.0, 1.05, 1.0, 1.1, 1.3, 1.25])           # lun..dim
trafic = rng_t.poisson(profil[None, None, :] * jour_facteur[None, :, None] * np.array([1.0, 1.05, 1.1, 1.2])[:, None, None])
```
`trafic` a pour forme `(4 semaines, 7 jours, 24 heures)`.

1. Vérifiez la forme. Calculez le total de visites par semaine et la croissance (%) de chaque semaine par rapport à la précédente.
2. Calculez le profil horaire moyen (24 valeurs) : à quelle heure le trafic est-il maximal en moyenne ?
3. Pour chaque jour de la semaine (lun…dim), donnez l'heure de pointe moyenne sur les 4 semaines.
4. Calculez le rapport « trafic moyen par jour de week-end / trafic moyen par jour de semaine ».
5. Trouvez la (semaine, jour, heure) avec le trafic maximal absolu (`argmax` + `unravel_index`).

### Exercice 4 — Portefeuille financier : algèbre matricielle
*Niveau : 🟠 exigeant*

`rendements` est un tableau `(250 jours, 5 actifs)` de rendements journaliers (colonnes dans l'ordre de `actifs`).

1. Calculez la moyenne et l'écart-type journaliers de chaque actif, puis annualisez (252 jours : moyenne × 252, écart-type × √252).
2. Calculez la matrice de covariance `C` (5×5) avec `np.cov(..., rowvar=False)` et la matrice de corrélation.
3. Quelle est la paire d'actifs la plus corrélée ? (parcourez uniquement le triangle supérieur strict.)
4. Pour un portefeuille à poids égaux `w = [0.2]*5`, calculez : le rendement quotidien moyen (`rendements @ w`), puis la variance via la formule matricielle `wᵀ C w`. Vérifiez qu'elle coïncide avec `np.var(rendements @ w, ddof=1)`.
5. Montrez que la volatilité du portefeuille est **inférieure** à la moyenne pondérée des volatilités individuelles (effet de diversification) et calculez le rapport.

### Exercice 5 — Moyenne mobile et cumul sans boucle
*Niveau : 🔵 intermédiaire*

1. Soit `x = np.random.default_rng(0).normal(100, 15, 100_000)`. Calculez la moyenne mobile sur une fenêtre de 7 valeurs :
   - d'abord avec une boucle Python `for`,
   - puis **sans boucle**, avec `np.cumsum`,
   - puis avec `np.convolve`.
2. Vérifiez que les trois résultats sont égaux (`np.allclose`).
3. Mesurez les temps d'exécution (`time.perf_counter`). Quel est le facteur d'accélération ?
4. Retrouvez, avec `np.maximum.accumulate`, le **maximum cumulé** de `x[:10]` et dites ce que cela donnerait sur un cours de bourse (« plus haut historique »).

### Exercice 6 — Régression linéaire à la main
*Niveau : 🟠 exigeant*

Générez des données :

```python
rng_l = np.random.default_rng(1)
X = rng_l.uniform(0, 10, size=(500, 2))                          # surface (x10 m²) et âge (années)
y = 3.0 * X[:, 0] - 2.0 * X[:, 1] + 5.0 + rng_l.normal(0, 1.5, 500)
```
1. Ajoutez une colonne de 1 à `X` (matrice de conception `A` de forme `(500, 3)`).
2. Retrouvez les coefficients avec les **équations normales** `β = (AᵀA)⁻¹ Aᵀ y`, via `np.linalg.solve`.
3. Retrouvez-les avec `np.linalg.lstsq`. Les deux résultats concordent-ils ?
4. Calculez les prédictions `ŷ`, les résidus, le RMSE et le R² (`1 − SSR/SST`).
5. Les résidus ont-ils une moyenne proche de 0 ? Un écart-type proche de 1,5 ?

### Exercice 7 — Affectation au centroïde le plus proche (vectorisation)
*Niveau : 🟠 exigeant*

Vous avez 5 000 clients décrits par 3 variables standardisées `Z` et 4 centres de segments `centres` :

```python
rng_k = np.random.default_rng(8)
Z = rng_k.normal(size=(5000, 3))
centres = rng_k.normal(size=(4, 3)) * 1.5
```
1. Écrivez une version avec **boucles** `for` qui affecte chaque client au centre le plus proche (distance euclidienne) — testez-la sur les 200 premiers clients seulement.
2. Écrivez la version **vectorisée** sur les 5 000 clients (via un tableau `(5000, 4, 3)` ou `(5000, 4)` de distances).
3. Vérifiez la concordance sur les 200 premiers.
4. Donnez l'effectif de chaque segment et les nouveaux centres (moyenne des clients de chaque segment) : c'est une itération de l'algorithme K-Means.
5. Mesurez le facteur d'accélération (extrapolez la boucle sur 5 000 clients).

---
# PARTIE II — Pandas : exploration, jointures, agrégations

### Exercice 8 — Explorer les trois tables
*Niveau : 🔵 intermédiaire*

1. Pour `clients`, `catalogue` et `commandes` : forme, types, valeurs manquantes, doublons sur la clé primaire (`id_client`, `id_produit`, `id_commande`).
2. Période couverte par `commandes` ; répartition des statuts (en %) ; répartition des modes de paiement.
3. Combien de commandes référencent un `id_client` **absent** de la table `clients` ? Un `id_produit` absent du catalogue ?
4. Combien de clients n'ont **aucune** commande ?
5. Résumez en 3 phrases dans une cellule Markdown les points d'attention de qualité de données.

### Exercice 9 — Jointures : construire la table d'analyse
*Niveau : 🟠 exigeant*

1. Construisez `cmd` en joignant `commandes` avec `catalogue` (sur `id_produit`) puis avec `clients` (sur `id_client`). Utilisez `how="left"` depuis `commandes` pour **ne perdre aucune ligne**. Contrôlez que `len(cmd) == len(commandes)` (utilisez `validate="many_to_one"`).
2. Précisez les suffixes si des noms de colonnes entrent en collision (`nom`, `categorie`, etc.).
3. Ajoutez `ca` (chiffre d'affaires de la ligne = `quantite × prix_unitaire`) et `marge` (= `quantite × (prix_unitaire − cout)`).
4. Que se passe-t-il si vous utilisez `how="inner"` pour la jointure avec `clients` ? Combien de lignes perdez-vous ? (Utilisez aussi `indicator=True` en `left` pour les identifier.)
5. Conservez dans `cmd_ok` uniquement les commandes dont la ville du client est connue et dont le statut est « Livrée ».

In [ ]:
# Table d'analyse de référence (solution partielle de l'exercice précédent, fournie pour la suite).
# Si vous avez réussi l'exercice précédent, vous pouvez la remplacer par la vôtre.
cmd = (commandes
       .merge(catalogue, on="id_produit", how="left", validate="many_to_one")
       .merge(clients, on="id_client", how="left", validate="many_to_one"))
cmd["ca"] = cmd["quantite"] * cmd["prix_unitaire"]
cmd["marge"] = cmd["quantite"] * (cmd["prix_unitaire"] - cmd["cout"])
cmd_ok = cmd[(cmd["statut"] == "Livrée") & cmd["ville"].notna()].copy()
print(cmd.shape, cmd_ok.shape)

### Exercice 10 — Chiffre d'affaires mensuel
*Niveau : 🔵 intermédiaire*

Avec `cmd_ok` (commandes livrées, clients connus) :
1. Calculez le CA, la marge et le nombre de commandes **par mois** (index = période mensuelle, ex. `2024-03`).
2. Calculez le taux de marge (marge/CA, en %) par mois.
3. Quels sont les 3 meilleurs mois en CA ? Que remarquez-vous (saisonnalité) ?
4. Ajoutez la croissance du CA (%) par rapport au mois précédent et indiquez le mois de plus forte croissance.
5. Quelle part du CA annuel représentent novembre et décembre ?

### Exercice 11 — GroupBy avancé : agrégations nommées et top N par groupe
*Niveau : 🟠 exigeant*

Avec `cmd_ok` :
1. Tableau par `categorie` : nombre de commandes, CA total, CA moyen par commande, marge totale, nombre de produits distincts vendus, nombre de clients distincts.
2. Le **top 3 des produits** (par CA) **dans chaque catégorie** : un DataFrame avec colonnes `categorie`, `nom`, `ca`, `rang`.
3. Pour chaque ville, la catégorie qui génère le plus de CA et sa part dans le CA de la ville.

### Exercice 12 — transform et filter
*Niveau : 🟠 exigeant*

Avec `cmd_ok` :
1. Ajoutez une colonne `ecart_moyenne_cat` = écart (en %) entre le montant de la ligne et le **montant moyen de sa catégorie** (utilisez `transform("mean")`).
2. Ajoutez le **z-score** du CA de la ligne **au sein de sa catégorie** : `(ca − moyenne_cat) / écart_type_cat`. Combien de lignes ont un z-score supérieur à 3 ?
3. Conservez uniquement les clients ayant passé **au moins 5 commandes livrées** (`groupby().filter`). Combien de clients et de lignes restent ?
4. Dans ce sous-ensemble de clients fidèles, quel est le panier moyen par commande, comparé à celui de l'ensemble ?

### Exercice 13 — Tableau croisé dynamique
*Niveau : 🔵 intermédiaire*

Avec `cmd_ok` :
1. Construisez avec `pivot_table` le CA par **mois** (lignes) et **catégorie** (colonnes), avec les totaux de ligne et de colonne (`margins=True`).
2. Pour chaque catégorie, quel est le mois du maximum de CA ? (`idxmax`)
3. Construisez le tableau des **parts de chaque catégorie** dans le CA du mois (chaque ligne somme à 100 %).
4. Construisez un pivot `ville × canal` avec le CA moyen par client (CA / clients distincts).

### Exercice 14 — Annulations et retours : analyser une proportion
*Niveau : 🟠 exigeant*

Avec `cmd` (toutes les commandes, quel que soit leur statut, clients connus uniquement) :
1. Calculez le taux d'annulation (% de lignes « Annulée ») et le taux de retour par **mode de paiement**. Quel mode a le taux d'annulation le plus élevé ?
2. Calculez le tableau croisé normalisé `canal × statut` (proportions en ligne).
3. La différence d'annulation entre « Cash à la livraison » et les autres est-elle **significative** ou due au hasard ? Faites un test du χ² d'indépendance entre `mode_paiement == Cash` et `statut == Annulée` avec `scipy.stats.chi2_contingency`. Interprétez la p-valeur.
4. Estimez le **manque à gagner** : CA des commandes annulées, et la part imputable au paiement en cash.

---
# PARTIE III — Nettoyage, transformation, fonctions personnalisées

### Exercice 15 — Nettoyer le fichier de ventes sale
*Niveau : 🟠 exigeant*

Le DataFrame `bruts` est un export désordonné (voir la présentation plus haut). Produisez `propre` tel que :

1. Les noms de colonnes sont en `snake_case` sans accent ni espace (`ref`, `date_vente`, `ville`, `produit`, `quantite`, `montant`).
2. Les doublons exacts sont supprimés (combien ?).
3. `ville` est normalisée (espaces supprimés, casse « Title ») et prend ses valeurs dans la liste `villes` (vérifiez avec `assert`).
4. `date_vente` est de type `datetime64` (deux formats présents : ISO `aaaa-mm-jj` et `jj/mm/aaaa`).
5. `montant` est numérique : retirer « FCFA » et les espaces ; « N/A » → `NaN`.
6. Ajoutez le `prix_unitaire` du catalogue (jointure sur le nom du produit).
7. `quantite` : les valeurs négatives sont invalides → `NaN`. Les quantités invalides ou manquantes sont **reconstituées** à partir de `montant / prix_unitaire` quand le montant est valide (sinon, médiane de la quantité du produit).
8. Les montants aberrants (supérieurs à 1 000 000) et les montants `NaN` sont recalculés par `quantite × prix_unitaire`.
9. Contrôles finaux : pas de `NaN`, pas de doublon sur `ref`, montants strictement positifs, et **`montant == quantite × prix_unitaire` pour toutes les lignes**. Affichez un petit rapport de nettoyage (nombre de lignes corrigées par type de problème).

### Exercice 16 — Segmentation RFM des clients
*Niveau : 🔴 synthèse*

La méthode **RFM** résume chaque client par : **R**écence (jours depuis la dernière commande), **F**réquence (nombre de commandes), **M**ontant (CA total). Avec `cmd_ok` :

1. Construisez `rfm` (un client par ligne) avec `recence` (par rapport au 31 décembre 2024), `frequence`, `montant`.
2. Attribuez à chacune des trois dimensions un score de 1 à 5 via `pd.qcut` (quintiles). *Attention : pour la récence, un score élevé doit signifier « récent »; pour la fréquence, beaucoup de clients ont la même valeur : utilisez `rank(method="first")` avant `qcut`.*
3. Définissez des segments : « Champions » (R ≥ 4, F ≥ 4, M ≥ 4), « Fidèles » (F ≥ 4), « À risque » (R ≤ 2 et F ≥ 3), « Nouveaux » (R ≥ 4 et F ≤ 2), « Autres ».
4. Pour chaque segment : nombre de clients, part du CA total, CA moyen par client.
5. Quel segment concentre la plus grande part du CA ? La règle des 80/20 de Pareto se vérifie-t-elle (part du CA des 20 % de clients les plus dépensiers) ?

### Exercice 17 — Fonction personnalisée de profilage
*Niveau : 🔵 intermédiaire*

1. Écrivez une fonction `profil(df, cle, col)` qui renvoie, pour chaque valeur de `cle`, un DataFrame avec : effectif, moyenne, médiane, 90ᵉ percentile, écart-type, et **coefficient de variation** (écart-type / moyenne, en %). Triez par moyenne décroissante.
2. Appliquez-la à `cmd_ok` : CA par `categorie`, puis CA par `canal`.
3. Écrivez une seconde fonction `outliers_iqr(serie, k=1.5)` qui renvoie un masque booléen des valeurs hors de `[Q1 − k·IQR, Q3 + k·IQR]`. Combien de lignes de `cmd_ok` sont des valeurs extrêmes de CA **au sein de leur catégorie** ? (utilisez `groupby(...).transform(outliers_iqr)`)
4. Ajoutez une docstring et un `assert` de contrôle de type d'entrée dans `profil`.

---
# PARTIE IV — Dates et séries temporelles

### Exercice 18 — Motifs hebdomadaires et moyenne mobile
*Niveau : 🔵 intermédiaire*

Avec `cmd_ok` :
1. Construisez la série quotidienne du CA sur **toute l'année 2024** (366 jours), en mettant 0 pour les jours sans vente (utilisez `resample("D")`, `reindex` ou `asfreq`).
2. Quel est le CA moyen par jour de la semaine (lundi…dimanche) ? Quel jour est le meilleur ?
3. Calculez la moyenne mobile sur 7 jours et sur 28 jours. Quel jour a la plus grande moyenne mobile 7 jours ?
4. Passez à une série hebdomadaire (`resample("W")`) : quelle est la semaine la plus forte et la variation (%) par rapport à la semaine précédente ?
5. Comparez le CA du premier et du dernier trimestre (`dt.quarter`).

### Exercice 19 — Capteur : trous, anomalies et profil journalier
*Niveau : 🟠 exigeant*

Avec `capteur` (consommation horaire, 90 jours) :
1. Combien de mesures manquent ? Où se situent les trous (début de chaque période manquante) ?
2. Comblez par interpolation temporelle (`interpolate(method="time")`) **uniquement les petits trous (≤ 3 heures)** et laissez `NaN` le grand trou de 30 heures. *Piège : `limit=3` comblerait les 3 premières heures du grand trou ; calculez d'abord la longueur de chaque période manquante.* Combien de valeurs restent manquantes ?
3. Détectez les **pics anormaux** : le profil journalier est périodique, donc comparez chaque mesure à la médiane des mesures **de la même heure et du même type de jour** (semaine / week-end). Une mesure est anormale si l'écart dépasse 6 écarts robustes (`1,4826 × MAD` des écarts). Combien de pics trouvez-vous ? (Attendu : 6, ceux qui ont été injectés.)
4. Remplacez ces pics par `NaN`, puis interpolez les petits trous.
5. Calculez le profil horaire moyen en distinguant semaine et week-end (heure en colonnes, deux lignes). À quelle heure le pic du soir intervient-il ?
6. Quelle est la corrélation entre `conso_kwh` et `temp_ext` ? Est-elle interprétable telle quelle ?

---
# PARTIE V — Optimisation et synthèse NumPy + Pandas

### Exercice 20 — Réduire la mémoire et accélérer
*Niveau : 🟠 exigeant*

1. Mesurez la mémoire de `cmd` avec `memory_usage(deep=True)`. Réduisez-la en : convertissant les colonnes à peu de modalités en `category`, en réduisant les entiers (`pd.to_numeric(downcast="integer")`) et les flottants (`float32` si approprié). Quel gain (%) ?
2. Vérifiez que les résultats d'analyse restent identiques (somme du CA par catégorie avant/après).
3. Comparez trois façons de calculer `ca = quantite × prix_unitaire` sur `cmd` : (a) `apply(axis=1)` avec une fonction, (b) une compréhension de liste sur `zip`, (c) l'opération vectorisée. Mesurez et classez.
4. Comparez deux façons de calculer une colonne `tranche` (`"petit"` si `ca < 5000`, `"moyen"` si `< 20000`, sinon `"gros"`) : `apply` avec `if/elif` contre `np.select`/`pd.cut`.

### Exercice 21 — Panier moyen : intervalle de confiance par bootstrap
*Niveau : 🔴 synthèse*

Combinez Pandas et NumPy. Pour chaque **canal d'acquisition**, estimez le **panier moyen par commande livrée** (`ca` dans `cmd_ok`) et son intervalle de confiance à 95 % par **bootstrap** :

1. Pour chaque canal, tirez 2 000 échantillons avec remise de la taille du groupe, calculez la moyenne de chacun (de façon vectorisée avec une matrice d'indices aléatoires `rng.integers(0, n, size=(2000, n))`).
2. L'IC à 95 % est donné par les percentiles 2,5 et 97,5 des 2 000 moyennes.
3. Rassemblez le tout dans un DataFrame (canal, n, moyenne, ic_bas, ic_haut) trié par moyenne.
4. Les IC de deux canaux se chevauchent-ils ? Que peut-on en conclure sur la différence entre canaux ?
5. Comparez avec l'IC théorique `moyenne ± 1,96 × écart-type/√n`.

### Exercice 22 — Analyse exploratoire complète : que raconte Kora en 2024 ?
*Niveau : 🔴 synthèse*

Rédigez un mini-rapport d'analyse (code + commentaires Markdown) sur `cmd`, `cmd_ok`, `clients` et `catalogue`, qui répond à ces questions :

1. **Vue d'ensemble** : CA livré, marge, panier moyen, nombre de clients actifs, taux d'annulation et de retour.
2. **Produits** : les 5 produits les plus rentables (marge totale) et les 5 pires ; existe-t-il des produits à forte marge unitaire mais peu vendus ?
3. **Clients** : quelle ville a le CA par client le plus élevé ? Quel canal d'acquisition fidélise le mieux (nombre moyen de commandes par client) ?
4. **Temps** : mois et jour de la semaine les plus forts.
5. **Corrélations** : matrice de corrélation entre `age`, nombre de commandes et CA total par client. Quelles relations sont notables ?
6. **Recommandations** : trois actions concrètes, chacune appuyée par un chiffre.

---

# CORRECTIONS ET EXPLICATIONS

> ⚠️ **Section de corrections.** Avant de lire une correction, essayez vraiment l'exercice, relisez le cours, testez des variantes. Comparez ensuite votre méthode à la solution : plusieurs solutions correctes existent souvent.

Chaque correction contient : une **solution exécutable**, l'**explication du raisonnement**, les **erreurs possibles** et, quand c'est pertinent, une **autre approche**.

Les solutions utilisent uniquement les jeux de données définis plus haut dans le notebook : exécutez d'abord toutes les cellules de préparation (menu *Run → Run All Cells* fonctionne, vos cellules de réponse vides n'affichent rien).

## Correction — Exercice 1 : Matrice de distances par broadcasting

In [ ]:
P = villes_coord[["lat", "lon"]].to_numpy()
diff = P[:, None, :] - P[None, :, :]                  # (6, 1, 2) - (1, 6, 2) -> (6, 6, 2)
D = 111 * np.sqrt((diff ** 2).sum(axis=-1))
print(np.allclose(D, D.T), np.allclose(np.diag(D), 0))
D_inf = D + np.diag(np.full(len(P), np.inf))          # on interdit « soi-même »
proche = D_inf.argmin(axis=1)
noms_v = villes_coord["ville"].to_numpy()
for i, j in enumerate(proche):
    print(f"{noms_v[i]:<14} -> {noms_v[j]:<14} {D[i, j]:6.1f} km")
i, j = np.unravel_index(D.argmax(), D.shape)
print("max :", noms_v[i], "-", noms_v[j], round(D[i, j], 1), "km")

**Raisonnement.** `P[:, None, :]` a la forme `(6, 1, 2)` et `P[None, :, :]` la forme `(1, 6, 2)` ; leur différence se diffuse en `(6, 6, 2)` : chaque paire `(i, j)` donne le vecteur `P[i] − P[j]`. La somme des carrés sur le dernier axe puis la racine donnent la norme. Pour exclure la diagonale (distance 0) lors du `argmin`, on y ajoute `+inf`. `np.unravel_index` convertit l'indice « aplati » du maximum en couple (ligne, colonne).

**Erreurs possibles.** Faire `D.argmin(axis=1)` sans neutraliser la diagonale : chaque ville serait sa propre voisine (distance 0). Oublier l'axe `-1` dans la somme.

**Autre approche.** `scipy.spatial.distance.cdist(P, P)` calcule la même matrice (mais le but ici est de maîtriser le broadcasting).

## Correction — Exercice 2 : Softmax par ligne et normalisation

In [ ]:
scores = np.array([[2.0, 1.0, 0.1, -1.0], [0.5, 2.5, 0.3, 0.0], [1.0, 1.0, 1.0, 1.0],
                   [-2., 0.0, 3.0, 1.0], [900., 905., 899., 902.]])
with np.errstate(over="ignore", invalid="ignore"):
    naif = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
print("naïf, dernière ligne :", naif[-1])                      # nan : exp(900) = inf
stable = np.exp(scores - scores.max(axis=1, keepdims=True))
stable = stable / stable.sum(axis=1, keepdims=True)
print(stable.round(3)); print(stable.sum(axis=1))
print(stable.argmax(axis=1), stable.max(axis=1).round(3))

**Raisonnement.** `keepdims=True` conserve l'axe réduit avec une taille 1 : `(5, 1)` se diffuse contre `(5, 4)`. `exp(900)` dépasse le plus grand flottant (`inf`), et `inf/inf = nan`. Soustraire le maximum de la ligne ne change pas le softmax (on multiplie numérateur et dénominateur par la même constante `exp(−max)`) mais garde les exponentielles dans une plage sûre. Cette astuce est utilisée dans toutes les bibliothèques de Machine Learning.

**Erreurs possibles.** Oublier `keepdims=True` : `exp(...).sum(axis=1)` est `(5,)` et la division échoue ou, pire, s'effectue contre le mauvais axe si la matrice est carrée.

## Correction — Exercice 3 : Tableau 3D : trafic web

In [ ]:
rng_t = np.random.default_rng(3)
heures = np.arange(24)
profil = 40 + 120 * np.exp(-((heures - 20) ** 2) / 10) + 60 * np.exp(-((heures - 12) ** 2) / 12)
jour_facteur = np.array([1.0, 1.0, 1.05, 1.0, 1.1, 1.3, 1.25])
trafic = rng_t.poisson(profil[None, None, :] * jour_facteur[None, :, None] * np.array([1.0, 1.05, 1.1, 1.2])[:, None, None])
print(trafic.shape)
tot_sem = trafic.sum(axis=(1, 2))
print(tot_sem, (np.diff(tot_sem) / tot_sem[:-1] * 100).round(1))
profil_moy = trafic.mean(axis=(0, 1))
print("heure de pointe globale :", profil_moy.argmax())
print("pointe par jour :", trafic.mean(axis=0).argmax(axis=1))           # (7, 24) -> argmax sur les heures
par_jour = trafic.sum(axis=2).mean(axis=0)                               # trafic quotidien moyen, 7 valeurs
print(round(par_jour[5:].mean() / par_jour[:5].mean(), 3))
print(np.unravel_index(trafic.argmax(), trafic.shape))

**Raisonnement.** Les axes sont (0 = semaine, 1 = jour, 2 = heure). On moyenne sur les axes qui **disparaissent** : le profil horaire moyen s'obtient avec `axis=(0, 1)`. Pour la pointe par jour, on moyenne d'abord sur les semaines (`axis=0`), ce qui donne `(7, 24)`, puis on prend l'`argmax` sur les heures (`axis=1`). `np.diff` calcule les différences successives, qu'on rapporte à la semaine précédente.

**Erreurs possibles.** Utiliser `axis=1` pour « par jour » alors que cet axe indexe les jours : on obtiendrait le total par heure. Pour éviter les erreurs : écrire sur un papier la forme et l'axe qui doit disparaître.

## Correction — Exercice 4 : Portefeuille financier : algèbre matricielle

In [ ]:
mu_a = rendements.mean(axis=0) * 252
vol_a = rendements.std(axis=0, ddof=1) * np.sqrt(252)
print(pd.DataFrame({"rend_annuel": mu_a, "vol_annuelle": vol_a}, index=actifs).round(3))
C = np.cov(rendements, rowvar=False)
R = np.corrcoef(rendements, rowvar=False)
iu = np.triu_indices(5, k=1)
k = np.abs(R[iu]).argmax()
print("paire la plus corrélée :", actifs[iu[0][k]], actifs[iu[1][k]], round(R[iu][k], 3))
w = np.full(5, 0.2)
var_pf = w @ C @ w
print(np.isclose(var_pf, np.var(rendements @ w, ddof=1)))
vol_pf = np.sqrt(var_pf) * np.sqrt(252)
vol_moy = (w * vol_a).sum()
print(round(vol_pf, 4), round(vol_moy, 4), round(vol_pf / vol_moy, 3))

**Raisonnement.** La variance d'un portefeuille de poids `w` est `wᵀ C w`. Comme les corrélations sont inférieures à 1, la volatilité du portefeuille est plus faible que la moyenne pondérée des volatilités : c'est la diversification. `np.triu_indices(5, k=1)` fournit les indices du triangle supérieur strict, ce qui évite de compter la diagonale (corrélation 1) et les paires en double.

**Erreurs possibles.** Chercher le maximum de `R` sans exclure la diagonale : il vaut 1 (un actif avec lui-même). Oublier `rowvar=False` : NumPy considérerait chaque *ligne* comme une variable et renverrait une matrice 250×250.

## Correction — Exercice 5 : Moyenne mobile et cumul sans boucle

In [ ]:
import time
x = np.random.default_rng(0).normal(100, 15, 100_000)
k = 7

t = time.perf_counter()
mm_boucle = np.array([x[i:i + k].mean() for i in range(len(x) - k + 1)])
t_boucle = time.perf_counter() - t

t = time.perf_counter()
c = np.cumsum(np.insert(x, 0, 0.0))
mm_cum = (c[k:] - c[:-k]) / k
t_cum = time.perf_counter() - t

t = time.perf_counter()
mm_conv = np.convolve(x, np.ones(k) / k, mode="valid")
t_conv = time.perf_counter() - t

print(np.allclose(mm_boucle, mm_cum), np.allclose(mm_boucle, mm_conv))
print(f"boucle {t_boucle:.3f}s | cumsum {t_cum:.5f}s | convolve {t_conv:.5f}s | gain ×{t_boucle / t_cum:.0f}")
print(np.maximum.accumulate(x[:10]).round(1))

**Raisonnement.** Astuce `cumsum` : la somme d'une fenêtre `[i, i+k)` vaut `c[i+k] − c[i]` où `c` est la somme cumulée (avec un 0 en tête). Un seul passage au lieu de `k` additions par fenêtre. `convolve` avec un noyau de `1/k` fait la même chose. `maximum.accumulate` renvoie le plus haut atteint jusqu'à chaque instant : sur un cours, c'est le « plus haut historique » (utile pour calculer le drawdown).

**Erreurs possibles.** Oublier le 0 initial dans le `cumsum` → fenêtres décalées d'un cran. `mode="valid"` est nécessaire pour ne garder que les fenêtres complètes.

## Correction — Exercice 6 : Régression linéaire à la main

In [ ]:
rng_l = np.random.default_rng(1)
X = rng_l.uniform(0, 10, size=(500, 2))
y = 3.0 * X[:, 0] - 2.0 * X[:, 1] + 5.0 + rng_l.normal(0, 1.5, 500)
A = np.column_stack([np.ones(len(X)), X])
beta_ne = np.linalg.solve(A.T @ A, A.T @ y)
beta_ls = np.linalg.lstsq(A, y, rcond=None)[0]
print(beta_ne.round(3), np.allclose(beta_ne, beta_ls))
y_hat = A @ beta_ls
res = y - y_hat
rmse = np.sqrt((res ** 2).mean())
r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(round(rmse, 3), round(r2, 4), round(res.mean(), 6), round(res.std(ddof=3), 3))

**Raisonnement.** Les équations normales minimisent `‖y − Aβ‖²` ; `solve(AᵀA, Aᵀy)` est préférable à `inv(AᵀA) @ Aᵀy` (plus stable, plus rapide). `lstsq` est encore plus robuste (décomposition en valeurs singulières). Les coefficients retrouvés sont proches des vrais paramètres (5, 3, −2) ; le RMSE ≈ 1,5 correspond au bruit injecté. La moyenne des résidus est nulle par construction quand le modèle contient une constante.

**Erreurs possibles.** Calculer l'inverse avec `np.linalg.inv` puis multiplier : fonctionne ici mais devient instable avec des variables corrélées. Oublier la colonne de 1 : le modèle passerait par l'origine et serait biaisé.

## Correction — Exercice 7 : Affectation au centroïde le plus proche (vectorisation)

In [ ]:
import time
rng_k = np.random.default_rng(8)
Z = rng_k.normal(size=(5000, 3))
centres = rng_k.normal(size=(4, 3)) * 1.5

t = time.perf_counter()
aff_boucle = []
for z in Z[:200]:
    d = [np.sqrt(((z - c) ** 2).sum()) for c in centres]
    aff_boucle.append(int(np.argmin(d)))
t_boucle = (time.perf_counter() - t) * 5000 / 200

t = time.perf_counter()
dist = np.linalg.norm(Z[:, None, :] - centres[None, :, :], axis=2)      # (5000, 4)
aff = dist.argmin(axis=1)
t_vec = time.perf_counter() - t

print(np.array_equal(aff[:200], aff_boucle))
print(np.bincount(aff, minlength=4))
nouveaux = np.array([Z[aff == j].mean(axis=0) for j in range(4)])
print(nouveaux.round(2))
print(f"accélération ≈ ×{t_boucle / t_vec:.0f}")

**Raisonnement.** `Z[:, None, :] − centres[None, :, :]` produit un tableau `(5000, 4, 3)` ; `norm(axis=2)` réduit les 3 coordonnées. L'`argmin(axis=1)` donne, pour chaque client, le centre le plus proche. La boucle exécute 5 000 × 4 petits calculs en Python ; la version NumPy fait tout en quelques appels C. Cette étape d'affectation est exactement le cœur de K-Means.

**Erreurs possibles.** Utiliser `.argmin()` sans axe : renvoie un seul indice sur le tableau aplati. Allouer un énorme tableau intermédiaire pour des millions de clients (mémoire !) : on découpe alors en blocs.

## Correction — Exercice 8 : Explorer les trois tables

In [ ]:
for nom, d, cle in [("clients", clients, "id_client"), ("catalogue", catalogue, "id_produit"), ("commandes", commandes, "id_commande")]:
    print(nom, d.shape, "| NA :", int(d.isna().sum().sum()), "| clés dupliquées :", int(d[cle].duplicated().sum()))
print(commandes["date"].min().date(), commandes["date"].max().date())
print((commandes["statut"].value_counts(normalize=True) * 100).round(1))
print((commandes["mode_paiement"].value_counts(normalize=True) * 100).round(1))
print("clients inconnus :", (~commandes["id_client"].isin(clients["id_client"])).sum())
print("produits inconnus :", (~commandes["id_produit"].isin(catalogue["id_produit"])).sum())
print("clients sans commande :", (~clients["id_client"].isin(commandes["id_client"])).sum())

**Raisonnement.** `isin` compare deux colonnes de clés : c'est le moyen le plus direct de détecter des références orphelines (intégrité référentielle). Les 15 commandes dont l'`id_client` est entre 9000 et 9099 n'ont pas de client correspondant : une jointure `inner` les ferait disparaître silencieusement.

**Erreurs possibles.** Ne contrôler que les valeurs manquantes (`isna`) et rater les problèmes d'intégrité entre tables.

## Correction — Exercice 9 : Jointures : construire la table d'analyse

In [ ]:
cmd_s = (commandes
       .merge(catalogue, on="id_produit", how="left", validate="many_to_one")
       .merge(clients, on="id_client", how="left", validate="many_to_one", indicator=True))
assert len(cmd_s) == len(commandes)
cmd_s["ca"] = cmd_s["quantite"] * cmd_s["prix_unitaire"]
cmd_s["marge"] = cmd_s["quantite"] * (cmd_s["prix_unitaire"] - cmd_s["cout"])
print(cmd_s["_merge"].value_counts())
inner = commandes.merge(clients, on="id_client", how="inner")
print("lignes perdues avec inner :", len(commandes) - len(inner))
cmd_ok_s = cmd_s[(cmd_s["_merge"] == "both") & (cmd_s["statut"] == "Livrée")].drop(columns="_merge").copy()
print(len(cmd_ok_s), cmd_ok_s.columns.tolist())

**Raisonnement.** `left` garde toutes les lignes de gauche ; `validate="many_to_one"` lève une erreur si la clé n'était pas unique à droite (ce qui multiplierait les lignes). `indicator=True` ajoute `_merge` (`both`, `left_only`) qui sert à repérer les lignes sans correspondance. Avec `inner`, les 15 commandes orphelines disparaissent sans avertissement : d'où l'importance de **comparer les effectifs avant/après**.

**Erreurs possibles.** Jointure sur des clés de types différents (`int` vs `str`) → aucune correspondance. Oublier que `catalogue` contient une colonne `nom` (nom du produit) qui pourrait entrer en collision avec d'autres colonnes.

## Correction — Exercice 10 : Chiffre d'affaires mensuel

In [ ]:
mois = cmd_ok["date"].dt.to_period("M")
t = cmd_ok.groupby(mois).agg(ca=("ca", "sum"), marge=("marge", "sum"), nb=("id_commande", "size"))
t["taux_marge_%"] = t["marge"] / t["ca"] * 100
t["croissance_%"] = t["ca"].pct_change() * 100
print(t.round(1))
print(t["ca"].nlargest(3))
print("plus forte croissance :", t["croissance_%"].idxmax())
print(round(t.loc["2024-11":"2024-12", "ca"].sum() / t["ca"].sum() * 100, 1), "%")

**Raisonnement.** `dt.to_period("M")` transforme les dates en périodes mensuelles, pratiques comme clé de groupe. `pct_change()` calcule la variation relative avec la ligne précédente. La saisonnalité visible : novembre/décembre (promotions de fin d'année) et un pic en juin. Pour des séries incomplètes, `resample("MS")` fait le même travail en comblant les mois vides.

**Erreurs possibles.** Grouper par `dt.month` : fusionnerait les mêmes mois de plusieurs années (ici une seule année, donc sans conséquence, mais piège en général). Calculer le taux de marge moyen comme moyenne des taux (faux) au lieu de `somme marge / somme CA`.

## Correction — Exercice 11 : GroupBy avancé : agrégations nommées et top N par groupe

In [ ]:
t1 = cmd_ok.groupby("categorie").agg(
    nb_cmd=("id_commande", "size"), ca=("ca", "sum"), ca_moyen=("ca", "mean"),
    marge=("marge", "sum"), nb_produits=("id_produit", "nunique"), nb_clients=("id_client", "nunique"))
print(t1.round(0))

ca_prod = cmd_ok.groupby(["categorie", "nom"], as_index=False)["ca"].sum()
ca_prod["rang"] = ca_prod.groupby("categorie")["ca"].rank(ascending=False, method="first").astype(int)
top3 = ca_prod[ca_prod["rang"] <= 3].sort_values(["categorie", "rang"])
print(top3)

vc = cmd_ok.groupby(["ville", "categorie"])["ca"].sum()
part = vc / vc.groupby("ville").transform("sum")
best = part.groupby("ville").idxmax()
print(pd.DataFrame({"categorie": [i[1] for i in best], "part_%": (part[best].values * 100).round(1)}, index=best.index))

**Raisonnement.** Pour un « top N par groupe », la méthode la plus lisible : calculer un **rang à l'intérieur du groupe** (`groupby().rank()`) puis filtrer `rang <= N`. Pour la dernière question : `transform("sum")` donne le total de la ville répété sur chaque ligne, ce qui permet de calculer la part ; `idxmax()` par groupe renvoie le couple (ville, catégorie) gagnant.

**Erreurs possibles.** Utiliser `.head(3)` après un tri global : on obtiendrait le top 3 *global*, pas par catégorie. Utiliser `rank` sans `method="first"` : des ex æquo peuvent donner plus de 3 lignes.

**Autre approche.** `ca_prod.sort_values("ca", ascending=False).groupby("categorie").head(3)` donne aussi le top 3 par groupe en une ligne.

## Correction — Exercice 12 : transform et filter

In [ ]:
d = cmd_ok.copy()
moy_cat = d.groupby("categorie")["ca"].transform("mean")
std_cat = d.groupby("categorie")["ca"].transform("std")
d["ecart_moyenne_cat"] = (d["ca"] / moy_cat - 1) * 100
d["z"] = (d["ca"] - moy_cat) / std_cat
print((d["z"] > 3).sum())
fideles = d.groupby("id_client").filter(lambda g: len(g) >= 5)
print(fideles["id_client"].nunique(), len(fideles))
print(round(fideles["ca"].mean(), 0), round(d["ca"].mean(), 0))

**Raisonnement.** `transform` renvoie un résultat **aligné ligne à ligne** (même longueur que d), idéal pour des calculs relatifs au groupe. `filter` conserve ou rejette des **groupes entiers** selon une condition sur le groupe. Les clients fidèles (≥ 5 commandes) ont un panier moyen proche du panier moyen global ici : la fidélité vient du nombre de commandes plus que du montant.

**Erreurs possibles.** Utiliser `agg("mean")` à la place de `transform("mean")` : résultat de taille différente → erreur ou mauvais alignement.

## Correction — Exercice 13 : Tableau croisé dynamique

In [ ]:
tp = cmd_ok.assign(mois=cmd_ok["date"].dt.to_period("M")).pivot_table(
    index="mois", columns="categorie", values="ca", aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
print(tp.round(0))
corps = tp.drop(index="Total", columns="Total")
print(corps.idxmax())
parts = corps.div(corps.sum(axis=1), axis=0) * 100
print(parts.round(1))
g = cmd_ok.groupby(["ville", "canal"]).agg(ca=("ca", "sum"), clients=("id_client", "nunique"))
print((g["ca"] / g["clients"]).unstack().round(0))

**Raisonnement.** `pivot_table` = `groupby` + `unstack` : on choisit lignes, colonnes, valeurs et fonction d'agrégation. Pour les parts, `div(..., axis=0)` divise **chaque ligne** par son total (alignement sur l'index). Avant de calculer les parts ou `idxmax`, on retire les marges « Total » qui fausseraient le résultat.

**Erreurs possibles.** `corps.div(corps.sum(axis=1))` sans `axis=0` : Pandas tente d'aligner sur les colonnes et renvoie des NaN.

## Correction — Exercice 14 : Annulations et retours : analyser une proportion

In [ ]:
from scipy.stats import chi2_contingency
c = cmd[cmd["ville"].notna()].copy()
taux = c.groupby("mode_paiement")["statut"].value_counts(normalize=True).unstack().mul(100).round(1)
print(taux)
print(pd.crosstab(c["canal"], c["statut"], normalize="index").mul(100).round(1))
tab = pd.crosstab(c["mode_paiement"] == "Cash à la livraison", c["statut"] == "Annulée")
chi2, p, dof, _ = chi2_contingency(tab)
print(tab, "\np-valeur =", p)
annule = c[c["statut"] == "Annulée"]
print(annule["ca"].sum(), round(annule.loc[annule["mode_paiement"] == "Cash à la livraison", "ca"].sum() / annule["ca"].sum() * 100, 1), "%")

**Raisonnement.** `value_counts(normalize=True)` dans un `groupby` donne les proportions de chaque statut par groupe. Le test du χ² compare les effectifs observés à ceux attendus sous l'indépendance : une p-valeur très faible (< 0,05) signifie que l'écart observé est improbable si le mode de paiement n'avait aucun lien avec l'annulation. Ici les données ont été construites avec un taux d'annulation plus élevé en cash, donc le test le détecte.

**Erreurs possibles.** Conclure à une causalité à partir d'une association : le cash est lié aux annulations, mais cela ne prouve pas que c'est le mode de paiement qui les cause (variables cachées possibles).

## Correction — Exercice 15 : Nettoyer le fichier de ventes sale

In [ ]:
p = bruts.copy()
p.columns = (p.columns.str.strip().str.lower()
             .str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
             .str.replace(" ", "_"))
n0 = len(p); p = p.drop_duplicates(); print("doublons supprimés :", n0 - len(p))

p["ville"] = p["ville"].str.strip().str.title()
assert set(p["ville"]) <= set(villes), set(p["ville"]) - set(villes)

p["date_vente"] = pd.to_datetime(p["date_vente"], format="mixed", dayfirst=True)
p["montant"] = pd.to_numeric(p["montant"].str.replace("FCFA", "", regex=False).str.replace(" ", "", regex=False)
                             .replace("N/A", np.nan), errors="coerce")
p = p.merge(catalogue[["nom", "prix_unitaire"]], left_on="produit", right_on="nom", how="left").drop(columns="nom")

montant_ok = p["montant"].notna() & (p["montant"] <= 1_000_000)
q_invalide = p["quantite"].isna() | (p["quantite"] <= 0)
depuis_montant = q_invalide & montant_ok
p.loc[depuis_montant, "quantite"] = p.loc[depuis_montant, "montant"] / p.loc[depuis_montant, "prix_unitaire"]
reste = p["quantite"].isna() | (p["quantite"] <= 0)
p.loc[reste, "quantite"] = p.groupby("produit")["quantite"].transform("median")[reste]
p["quantite"] = p["quantite"].round().astype(int)

p.loc[~montant_ok, "montant"] = p.loc[~montant_ok, "quantite"] * p.loc[~montant_ok, "prix_unitaire"]
p["montant"] = p["montant"].astype(int)
print({"quantités reconstituées": int(depuis_montant.sum()), "montants recalculés": int((~montant_ok).sum())})

assert not p.isna().any().any()
assert not p["ref"].duplicated().any()
assert (p["montant"] > 0).all()
assert (p["montant"] == p["quantite"] * p["prix_unitaire"]).all()
print(p.dtypes); print(p.head())

**Raisonnement.** Le nettoyage est une succession d'opérations simples, chacune **vérifiable** : on contrôle avec des `assert`. `format="mixed"` + `dayfirst=True` gèrent les deux formats de date. Le point clé : une quantité invalide n'est pas imputée « à l'aveugle » (médiane) alors qu'on dispose d'une **contrainte métier** `montant = quantité × prix` : on la reconstitue exactement à partir du montant valide. On ne se rabat sur la médiane du produit (`groupby().transform("median")`) qu'en dernier recours. On calcule les masques (`montant_ok`, `q_invalide`) **avant** de modifier les colonnes pour que chaque correction s'appuie sur l'état d'origine.

**Erreurs possibles.** Imputer la quantité par la médiane alors qu'elle est déductible du montant : on introduit une erreur (la contrainte `montant == quantité × prix` échoue). Corriger les montants aberrants avant d'avoir reconstitué les quantités. Supprimer `title()` sans `strip()`.

## Correction — Exercice 16 : Segmentation RFM des clients

In [ ]:
fin = pd.Timestamp("2024-12-31")
rfm = cmd_ok.groupby("id_client").agg(derniere=("date", "max"), frequence=("id_commande", "size"), montant=("ca", "sum"))
rfm["recence"] = (fin - rfm["derniere"]).dt.days
rfm["R"] = pd.qcut(rfm["recence"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)    # petit = récent = score élevé
rfm["F"] = pd.qcut(rfm["frequence"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["montant"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["segment"] = np.select(
    [(rfm.R >= 4) & (rfm.F >= 4) & (rfm.M >= 4), rfm.F >= 4, (rfm.R <= 2) & (rfm.F >= 3), (rfm.R >= 4) & (rfm.F <= 2)],
    ["Champions", "Fidèles", "À risque", "Nouveaux"], default="Autres")
seg = rfm.groupby("segment").agg(clients=("montant", "size"), ca=("montant", "sum"), ca_moyen=("montant", "mean"))
seg["part_ca_%"] = seg["ca"] / seg["ca"].sum() * 100
print(seg.sort_values("ca", ascending=False).round(1))
tri = rfm["montant"].sort_values(ascending=False)
top20 = tri.head(int(len(tri) * 0.2)).sum() / tri.sum() * 100
print("part du CA des 20 % de clients les plus dépensiers :", round(top20, 1), "%")

**Raisonnement.** `qcut` découpe en classes d'effectifs égaux ; sur des variables avec beaucoup d'ex æquo (fréquence), il échoue (bornes dupliquées) : on le fait donc sur le **rang** `rank(method="first")`. L'ordre des étiquettes pour la récence est inversé car une faible récence (dernier achat récent) est positive. `np.select` applique les règles dans l'ordre de priorité. Le calcul de Pareto trie les montants et somme le haut de la liste : ici les 20 % de clients les plus dépensiers réalisent environ 46 % du CA, loin des 80 % de la « règle » : cette règle est une heuristique, pas une loi.

**Erreurs possibles.** Donner le même sens aux trois scores sans réfléchir à la récence. Oublier que `qcut` peut produire moins de classes que demandé si les valeurs sont trop concentrées.

## Correction — Exercice 17 : Fonction personnalisée de profilage

In [ ]:
def profil(df, cle, col):
    # Résumé statistique de `col` pour chaque modalité de `cle`.
    assert isinstance(df, pd.DataFrame) and {cle, col} <= set(df.columns), "colonnes manquantes"
    g = df.groupby(cle)[col]
    out = pd.DataFrame({"n": g.size(), "moyenne": g.mean(), "mediane": g.median(),
                        "p90": g.quantile(0.9), "ecart_type": g.std()})
    out["cv_%"] = out["ecart_type"] / out["moyenne"] * 100
    return out.sort_values("moyenne", ascending=False)

print(profil(cmd_ok, "categorie", "ca").round(1))
print(profil(cmd_ok, "canal", "ca").round(1))

def outliers_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

mask = cmd_ok.groupby("categorie")["ca"].transform(outliers_iqr)
print(int(mask.sum()), "valeurs extrêmes sur", len(cmd_ok))

**Raisonnement.** Une fonction réutilisable évite de dupliquer du code et rend l'analyse reproductible. `groupby(...).transform(f)` applique `f` à chaque groupe et réaligne le résultat sur les lignes d'origine : parfait pour un masque d'outliers « relatif à la catégorie ». Les outliers doivent être jugés **dans leur contexte** : un panier de 60 000 FCFA est normal en Électronique mais extrême en Alimentation.

**Erreurs possibles.** Utiliser `apply` ligne par ligne à la place de fonctions vectorisées. Détecter les outliers globalement alors que les catégories ont des échelles très différentes.

## Correction — Exercice 18 : Motifs hebdomadaires et moyenne mobile

In [ ]:
quot = cmd_ok.set_index("date")["ca"].resample("D").sum()
quot = quot.reindex(pd.date_range("2024-01-01", "2024-12-31"), fill_value=0)
print(len(quot))
print(quot.groupby(quot.index.day_name()).mean().round(0).sort_values(ascending=False))
mm7 = quot.rolling(7).mean(); mm28 = quot.rolling(28).mean()
print(mm7.idxmax().date(), round(mm7.max(), 0))
hebdo = cmd_ok.set_index("date")["ca"].resample("W").sum()
print(hebdo.idxmax().date(), hebdo.pct_change().mul(100).round(1).tail(3).tolist())
tri = quot.groupby(quot.index.quarter).sum()
print(tri)

**Raisonnement.** `resample` regroupe par période ; `reindex` sur une plage complète garantit que les jours sans vente apparaissent (avec 0, et pas absents). `rolling(7).mean()` est une moyenne glissante (les 6 premières valeurs sont NaN). Les groupes `day_name()` montrent la saisonnalité hebdomadaire (week-end plus fort, par construction des données). Attention : la dernière « semaine » de `resample("W")` est **incomplète** (le 31 décembre tombe un mardi), sa variation négative est donc un artefact à ne pas interpréter.

**Erreurs possibles.** Oublier les jours sans vente : une moyenne mobile sur 7 *lignes* ne correspond plus à 7 *jours*. Utiliser `.dt` sur un `DatetimeIndex` (il se dit `index.day_name()` directement).

## Correction — Exercice 19 : Capteur : trous, anomalies et profil journalier

In [ ]:
s = capteur["conso_kwh"].copy()
na = s.isna()
debuts = s.index[na & ~na.shift(fill_value=False)]
print("manquantes :", int(na.sum()), "| débuts de trous :", [d.strftime("%d/%m %Hh") for d in debuts])

# Longueur de chaque période manquante, reportée sur ses lignes
groupe = na.ne(na.shift()).cumsum()
longueur = na.groupby(groupe).transform("sum")
petits = na & (longueur <= 3)
s2 = s.copy()
s2[petits] = s.interpolate(method="time")[petits]
print("restantes :", int(s2.isna().sum()))                 # 30 : le grand trou est conservé

cle = [s.index.dayofweek >= 5, s.index.hour]
attendu = s.groupby(cle).transform("median")
ecart = s - attendu
sigma = 1.4826 * ecart.abs().median()
pics = ecart > 6 * sigma
print("pics détectés :", int(pics.sum()), "| seuil :", round(6 * sigma, 2), "kWh")

s_clean = s.mask(pics)
longueur2 = s_clean.isna().groupby(s_clean.isna().ne(s_clean.isna().shift()).cumsum()).transform("sum")
petits2 = s_clean.isna() & (longueur2 <= 3)
s_clean[petits2] = s_clean.interpolate(method="time")[petits2]

prof = s_clean.groupby([s_clean.index.dayofweek >= 5, s_clean.index.hour]).mean().unstack()
prof.index = ["semaine", "week-end"]
print(prof.round(2).to_string())
print("pic du soir (semaine) :", prof.loc["semaine", 15:].idxmax(), "h")
print(round(capteur["conso_kwh"].corr(capteur["temp_ext"]), 3))

**Raisonnement.** Pour traiter des trous différemment selon leur longueur, on repère les **séquences consécutives** de NaN avec l'astuce `na.ne(na.shift()).cumsum()` (un numéro par séquence), puis `transform("sum")` donne la longueur de la séquence sur chacune de ses lignes. Pour détecter des anomalies dans une série **périodique**, il faut comparer à ce qui est *attendu à ce moment du cycle* (médiane par heure et type de jour), sinon les pics normaux du soir seraient signalés comme anormaux. Le MAD (écart absolu médian) est une mesure de dispersion robuste aux anomalies elles-mêmes. La corrélation avec la température est faible et ne signifie pas grand-chose telle quelle : les deux variables ont chacune un cycle journalier (mais décalé), ce qui crée une corrélation de saisonnalité ; il faudrait comparer les écarts au profil attendu.

**Erreurs possibles.** Combler un trou de 30 heures par interpolation linéaire invente des données. Utiliser `interpolate(limit=3)` en croyant ne combler que les petits trous. Détecter les pics avec un seuil global (moyenne ± 3σ) ou une médiane glissante 24 h : le cycle journalier créerait des dizaines de fausses alertes.

## Correction — Exercice 20 : Réduire la mémoire et accélérer

In [ ]:
import time
avant = cmd.memory_usage(deep=True).sum() / 1e6
opt = cmd.copy()
for c in ["categorie", "nom", "ville", "canal", "statut", "mode_paiement"]:
    opt[c] = opt[c].astype("category")
for c in opt.select_dtypes("integer").columns:
    opt[c] = pd.to_numeric(opt[c], downcast="integer")
for c in opt.select_dtypes("float").columns:
    opt[c] = pd.to_numeric(opt[c], downcast="float")
apres = opt.memory_usage(deep=True).sum() / 1e6
print(f"{avant:.2f} Mo -> {apres:.2f} Mo ({(1 - apres / avant) * 100:.0f} % de gain)")
print(np.allclose(cmd.groupby("categorie", observed=True)["ca"].sum(), opt.groupby("categorie", observed=True)["ca"].sum()))

def chrono(f):
    t = time.perf_counter(); f(); return time.perf_counter() - t
ta = chrono(lambda: cmd.apply(lambda r: r["quantite"] * r["prix_unitaire"], axis=1))
tb = chrono(lambda: [q * p for q, p in zip(cmd["quantite"], cmd["prix_unitaire"])])
tc = chrono(lambda: cmd["quantite"] * cmd["prix_unitaire"])
print(f"apply {ta:.4f}s | zip {tb:.4f}s | vectorisé {tc:.5f}s")

def tr(v): return "petit" if v < 5000 else ("moyen" if v < 20000 else "gros")
t1 = chrono(lambda: cmd["ca"].apply(tr))
t2 = chrono(lambda: np.select([cmd["ca"] < 5000, cmd["ca"] < 20000], ["petit", "moyen"], "gros"))
print(f"apply {t1:.4f}s | select {t2:.5f}s")

**Raisonnement.** Le type `category` stocke chaque modalité une fois et référence des codes entiers : gain majeur pour des colonnes texte répétitives. `downcast` choisit le plus petit entier suffisant. La vectorisation exécute la boucle en C : c'est typiquement 50 à 1000 fois plus rapide qu'`apply(axis=1)`. Comparer **les résultats** (pas seulement la vitesse) est indispensable pour valider une optimisation.

**Erreurs possibles.** Convertir en `float32` des montants financiers qui exigent la précision : ici, des entiers FCFA, aucun problème, mais en général vérifiez. Optimiser sans mesurer.

## Correction — Exercice 21 : Panier moyen : intervalle de confiance par bootstrap

In [ ]:
rng_bs = np.random.default_rng(42)
lignes = []
for canal, g in cmd_ok.groupby("canal"):
    v = g["ca"].to_numpy(dtype=float)
    n = len(v)
    moyennes = v[rng_bs.integers(0, n, size=(2000, n))].mean(axis=1)
    lo, hi = np.percentile(moyennes, [2.5, 97.5])
    se = v.std(ddof=1) / np.sqrt(n)
    lignes.append((canal, n, v.mean(), lo, hi, v.mean() - 1.96 * se, v.mean() + 1.96 * se))
res = pd.DataFrame(lignes, columns=["canal", "n", "moyenne", "ic_bas", "ic_haut", "th_bas", "th_haut"]).set_index("canal")
print(res.sort_values("moyenne", ascending=False).round(0))

**Raisonnement.** Le bootstrap rééchantillonne les données **avec remise** pour approximer la distribution de la statistique. L'indexation `v[matrice_d_indices]` produit une matrice `(2000, n)` de rééchantillons ; `.mean(axis=1)` donne les 2 000 moyennes. Pour ces tailles d'échantillon, bootstrap et IC théorique concordent. Si les IC de deux canaux se chevauchent largement, on ne peut pas affirmer que leurs paniers moyens diffèrent vraiment ; un test (Welch) ou un bootstrap sur la différence serait plus rigoureux.

**Erreurs possibles.** Boucler 2 000 fois en Python au lieu de la matrice d'indices (lent). Confondre l'IC de la **moyenne** (étroit) avec l'intervalle de dispersion des **valeurs individuelles** (large).

## Correction — Exercice 22 : Analyse exploratoire complète : que raconte Kora en 2024 ?

In [ ]:
livre = cmd_ok
print("CA :", int(livre["ca"].sum()), "| marge :", int(livre["marge"].sum()),
      "| panier moyen :", round(livre["ca"].mean()), "| clients actifs :", livre["id_client"].nunique())
base_ok = cmd[cmd["ville"].notna()]
print((base_ok["statut"].value_counts(normalize=True) * 100).round(1).to_dict())

prod = livre.groupby("nom").agg(marge=("marge", "sum"), qte=("quantite", "sum"), marge_unit=("marge", "mean"))
print(prod["marge"].nlargest(5)); print(prod["marge"].nsmallest(5))
print(prod[(prod["marge_unit"] > prod["marge_unit"].quantile(0.75)) & (prod["qte"] < prod["qte"].quantile(0.25))])

par_client = livre.groupby("id_client").agg(ca=("ca", "sum"), nb=("id_commande", "size")).join(clients.set_index("id_client")[["ville", "canal", "age"]])
print(par_client.groupby("ville")["ca"].mean().round(0).sort_values(ascending=False))
print(par_client.groupby("canal")["nb"].mean().round(2).sort_values(ascending=False))

print(livre.groupby(livre["date"].dt.month)["ca"].sum().idxmax(), livre.groupby(livre["date"].dt.day_name())["ca"].sum().idxmax())
print(par_client[["age", "nb", "ca"]].corr().round(2))

**Raisonnement.** Ce type de rapport enchaîne : agrégats globaux, classements, croisements (ville/canal), temps, corrélations. Vos recommandations doivent s'appuyer sur ces chiffres (par exemple : « concentrer le budget sur le canal X qui a le meilleur nombre de commandes par client », « traiter le paiement à la livraison dont le taux d'annulation est plus de deux fois supérieur »). La corrélation entre le nombre de commandes et le CA est mécaniquement forte ; celle avec l'âge est ici faible, ce qui est lui-même une information.

**Erreurs possibles.** Présenter des corrélations triviales comme des découvertes ; ne pas préciser le périmètre (commandes livrées vs toutes) pour chaque chiffre.